# Japan Train Coverage

Identifying underserved areas by Japanese train stations

The purpose of this project is to help identify areas in Japan that are underserved or overloaded based on the relationship between train station coverage and population density. The goal is to identify areas that do not have an appropriate number of stations, stations in wrong relative location for population centers, or general outliers. It may provide insight into where additional train stations may be beneficial in the future and help provide public transport to those areas.

This analysis may be performed through several methods either by dividing up the country based on prefectures, grids, or based on the areas surrounding the train stations themselves. This may provide different point of views to identify such underserved areas. Finally, analysis of these outliers may be performed not just through GIS mapping methods but using cluster analysis to identify outliers based on population or identifying the common relationship of population to number of stations.

### Install Requirements

In [ ]:
from matplotlib.pyplot import yscale
from matplotlib.ticker import LogLocator
%pip install geopandas overpy pandas matplotlib folium mapclassify geodatasets fastparquet rasterio h3pandas rasterstats seaborn osmium

### Obtain the list of train stations

The first step is to pull a list of train stations using the Overpy API. This will create a parquet to save the station list including station name, latitude, and longitude.

In [ ]:
# Import Stations Helper

# Use the overpass api query
# from helpers.stations_api import list_stations

# Parse local osm files instead of using the api, this query can take longer but allow for historical queries
from helpers.stations_local import list_stations

### Map the train stations

Map the train stations for a quick visualization.

In [ ]:
import folium

# Call list_stations with current year
stations_list_2025 = list_stations(2025)

# Define folium map - centered on Toyama
m = folium.Map(location=[36.70,137.21], zoom_start=6)

# Iterate over station list and map on folium
for index, row in stations_list_2025.iterrows():
	folium.Marker(
		location=[row['latitude'], row['longitude']],
		popup=row['name'],
		tooltip=row['name']
	).add_to(m)

# Map the stations list
m

### Read in population data

Next step is to read in the population map and convert to a usable format.

Population data is obtained from: https://hub.worldpop.org/geodata/summary?id=77823

The resolution of the dataset was determined as follows:
1. Download the dataset TIF
2. Install GDAL
```bash
conda install -c conda-forge gdal
```
3. Check resolution of the dataset
```bash
gdalinfo jpn_pop_2025_CN_1km_R2025A_UA_v1.tif
```

The dataset jpn_pop_2025_CN_1km_R2025A_UA_v1.tif is population per 1km grid.

In [ ]:
import os
import pandas as pd
import rasterio
from rasterio import mask

# Read in the population tif
raster_pop_2025 = rasterio.open('public-datasets/jpn_pop_2025_CN_1km_R2025A_UA_v1.tif')
print(f'Population CRS: {raster_pop_2025.crs}')

### Map the population data

Map the population data for a quick visualization.

In [ ]:
from matplotlib import pyplot

pyplot.imshow(raster_pop_2025.read(1), cmap='Reds')
pyplot.show()

### Divide Japan into Hex Grids

Divide japan into hex grids (H3) to perform analysis.

Use the following coastline shp file: https://data.humdata.org/dataset/cod-xa-jpn to set the boundaries.

In [ ]:
import h3pandas
import geopandas as gpd

# Download the Geometry file for bounding coastlines
# https://data.humdata.org/dataset/cod-xa-jpn

# Load the Geometry file - EPSG:4326
coastline = gpd.read_file("public-datasets/jpn_adm_2019_shp/jpn_admbnda_adm0_2019.shp")

# Set the grid size, 5 sets rougly 1700 grids
resolution = 5

# Convert the dataframe into h3 pandas
coastline_h3 = coastline.h3.polyfill_resample(resolution)

# Sort the h3 in same order every time to ensure order for grid numbers vs population stays same
coastline_h3 = coastline_h3.sort_values('h3_polyfill')

# Clean column names before saving, remove any newlines for h3_polyfill
coastline_h3.columns = coastline_h3.columns.str.strip()

# Save polygons h3 as shp file for use later
coastline_h3.to_file("data/h3-polygons.shp", driver='ESRI ShapeFile')

# Generate folium map for coastline clipping and box - centered on Toyama
m = folium.Map(location=[36.70,137.21], tiles="Cartodb dark_matter", zoom_start=6)
folium.GeoJson(coastline_h3).add_to(m)

# Map
m

### Create a class for storing grid data

Create a class for storing grid data including the following and it's methods:
- Grid ID
- Grid Centroid
- Distance from the centroid to nearest station
- Geography polygon
- Population
- Station count
- Station list

### Assign Train stations to grids

Build a dictionary of grids with number of stations within the grid.

In [ ]:
from helpers.grids import GridInfo
from helpers.grids import grid_assign_stations

# Grid object list
grid_array_2025 = []

# Assign stations to the 2025 grid_array objects
grid_array_2025 = grid_assign_stations(coastline_h3, stations_list_2025, grid_array_2025)

# Save as dataframe for DataSpell inspect - just returning station count
grid_array_df = pd.DataFrame.from_records([s.get_station_count()] for s in grid_array_2025)
print(f'Created grid array with count: {len(grid_array_2025)}')

### Plot the station count

Quick visualization for station counts by grid_array

In [ ]:
grid_array_df_out = grid_array_df.copy()
grid_array_df_out

### Assign population to the grid

Clip the population from the TIF to the geometry of the h3 cell

In [ ]:
import rasterio.mask

# Mask the data using the h3 shapes
with rasterio.open('public-datasets/jpn_pop_2025_CN_1km_R2025A_UA_v1.tif') as src:

    # Check projection
    h3_projection = coastline_h3.iloc[:1].to_crs(src.crs)
    out_image, out_transform = rasterio.mask.mask(src, h3_projection.geometry, crop=True, all_touched=True)
    out_meta = src.meta

# Show the masked image
out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

# Write test tif
with rasterio.open("data/masked-pop.tif", "w", **out_meta) as dest:
    dest.write(out_image)

# Map the clipped raster
pyplot.figure(figsize=(5,5))
pyplot.imshow(out_image[0], cmap='Reds')
pyplot.title("single h3 cell clip")
pyplot.axis("off")
pyplot.show()

### Raster statistics for each zone

Sum the population in each zone

In [ ]:
from helpers.population import population_assign_grid_array

# Call function for current 2025 raster data
raster = 'public-datasets/jpn_pop_2025_CN_1km_R2025A_UA_v1.tif'

# Call to population_assign_grid_array to update grid_array_2025 with its population values
total_pop_2025 = population_assign_grid_array(grid_array_2025, raster)

# Print the total population for 2025
print(f'total population: {total_pop_2025}')

### Check grid alignment

Check that h3_id and population are matching properly by using a known control island of Niijima and it's population.

In [ ]:
# Population asserts for specific grid check to check order and clipping is right

# Pull grid population for specific id
for obj in grid_array_2025:

    # Control grid (Niijima - 新島村) - h3_id = 852f59d3fffffff
    if obj.get_h3_id() == '852f59d3fffffff':

        print("Control grid found: (Niijima - 新島村) - h3_id = 852f59d3fffffff")

        # Get population for the item
        population = obj.get_population()

        # Assert - 2020 population = 2441 (per Wikipedia)
        # Population either changed from 2020 to 2025 where the raster is or clipping is wrong
        assert population == 2420

else:
    assert "failed to find control grid"

### Check grid array

Check grid array accuracy

In [ ]:
print(grid_array_2025[1].get_population())

### Map the grid_array data

In [ ]:
# Define folium map - centered on Toyama
m = folium.Map(location=[36.70,137.21], zoom_start=6)

# Iterate over station list and map population counts on folium for each grid
for grid_obj in grid_array_2025:
    lat,lon = grid_obj.get_centroid()
    folium.Marker(
		location=[lat,lon],
		popup=grid_obj.get_h3_id(),
		tooltip=grid_obj.get_population(),
	).add_to(m)

# Add data to folium map
folium.GeoJson(coastline_h3).add_to(m)

# Map the stations list
m

### Calculate distance to nearest station

If the station count is empty, which is the next nearest station distance.

Distance shows the distance from centroid of the grid to the nearest station.

In [ ]:
from helpers.grids import assign_grid_centroid_neighbors

# Call to assign centroids neighbors to grid_array_2025
grid_array_2025 = assign_grid_centroid_neighbors(stations_list_2025, grid_array_2025)

### Create Master Unified Base Table

Create a unified dataframe and parquet for easier parsing.

In [ ]:
from helpers.grids import grid_array_create_df

final_df_2025 = grid_array_create_df(grid_array_2025)
final_df_2025_copy = final_df_2025.copy()

### Analysis

Analysis on the following points to gather further insights (below in relationship to population)
- Centroid distance to nearest station (centroid_to_station_km)
- Population per grid (population)
- Station count per grid (station_count)
- Station count per capita (per_capita_10k)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Read parquet if already exists
if os.path.exists('data/final-station-table.parquet'):
    final_df_2025 = pd.read_parquet('data/final-station-table.parquet')

# Pull population as x data
population = final_df_2025['population']
grids = final_df_2025['h3_id']
station_count = final_df_2025['station_count']

# Plot the population for each grid, this order should not change
# Fixed order changing every time, this is set by the sort in the h3_id grid definition
plt.xlabel("grid numbers")
plt.ylabel("population")
plt.ticklabel_format(useOffset=False, style='plain')
plt.plot(grids, population)

In [ ]:
# Map the relationship between population and station count
x = population
y = station_count
data = list(zip(population, station_count))
kmeans = KMeans(n_clusters=4)
kmeans.fit(data)
plt.xlabel("population per grid")
plt.ylabel("station count per grid")
plt.ticklabel_format(useOffset=False, style='plain')
plt.scatter(x, y, c=kmeans.labels_, cmap='tab10')

In [ ]:
# Centroid distance to nearest station (centroid_to_station_km)
centroid_to_station_km = final_df_2025['centroid_to_station_km']

# Map the relationship between population and centroid distance to nearest station
x = population
y = centroid_to_station_km
data = list(zip(population, centroid_to_station_km))
kmeans = KMeans(n_clusters=4)
kmeans.fit(data)
plt.xlabel("population per grid")
plt.ylabel("distance to nearest station in km")
plt.ticklabel_format(useOffset=False, style='plain')
plt.scatter(x, y, c=kmeans.labels_, cmap='tab10')

In [ ]:
# Remove the grids wit population > 100,000
population_no_empty = final_df_2025.query('population >= 100000')['population']

# Remove centroids without population
centroid_to_station_km_no_empty = final_df_2025.query('population >= 100000')['centroid_to_station_km']

# Map the relationship between population and centroid distance to nearest station
x = population_no_empty
y = centroid_to_station_km_no_empty
data = list(zip(population_no_empty, centroid_to_station_km_no_empty))
kmeans = KMeans(n_clusters=4)
kmeans.fit(data)
plt.xlabel("population per grid > 100,000")
plt.ylabel("distance to nearest station in km")
plt.ticklabel_format(useOffset=False, style='plain')
plt.scatter(x, y, c=kmeans.labels_, cmap='tab10')

In [ ]:
# Number of stations per capita, remove NaN and population >= 1,000
valid_df = final_df_2025.query("population >= 1000 and per_capita_10k == per_capita_10k")

# Pull out population and new per capita
per_cap_valid = valid_df['per_capita_10k']
pop_valid = valid_df['population']

# Map the relationship between population and station count
x = pop_valid
y = per_cap_valid
data = list(zip(x, y))
kmeans = KMeans(n_clusters=4)
kmeans.fit(data)
plt.xlabel("population per grid")
plt.ylabel("station count per 10,000 people")
plt.ticklabel_format(useOffset=False, style='plain')
plt.scatter(x, y, c=kmeans.labels_, cmap='tab10')

In [ ]:
# major city centers

# Number of stations per capita, remove NaN and population >= 100,000
valid_df = final_df_2025.query("population >= 100000 and per_capita_10k == per_capita_10k")

# Pull out population and new per capita
per_cap_valid = valid_df['per_capita_10k']
pop_valid = valid_df['population']

# Map the relationship between population and station count
x = pop_valid
y = per_cap_valid
data = list(zip(pop_valid, per_cap_valid))
kmeans = KMeans(n_clusters=4)
kmeans.fit(data)
plt.xlabel("population per grid")
plt.ylabel("station count per 10,000 people, grid is > 100,000 people")
plt.ticklabel_format(useOffset=False, style='plain')
plt.scatter(x, y, c=kmeans.labels_, cmap='tab10')

In [ ]:
import seaborn as sns

# Show correlation metrics
corr_matrix = final_df_2025_copy.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

### Heatmap low per capita areas

Draw a heatmap of low per capita areas to show underserved grids.

In [ ]:
from folium.plugins import HeatMap
import numpy as np

# Define folium map - centered on Toyama
m = folium.Map(location=[36.70,137.21], zoom_start=6)

# Define data array for heatmap
data = []

# Iterate over station list and map on folium
for index, row in final_df_2025.iterrows():
    centroid = row['centroid']
    picked = row['per_capita_10k']
    
    # parse out rows
    latitude, longitude = centroid

    # Parse out NaN
    if pd.notna(picked):
        
        # Create data
        data.append([latitude, longitude, picked])

# Add to map
HeatMap(data, blur=10, min_opacity=.4).add_to(m)
    
# Map the stations list
m

### Pattern Evaluation



In [ ]:
# Map urban vs rural areas

# Which grids have at least 1 station (classification)
from helpers.grids import grid_assign_classification

classify_final_df, data, m = grid_assign_classification(final_df_2025)

# Add to map
HeatMap(data).add_to(m)

# Map the stations list
m

In [ ]:
import seaborn as sns

# Show correlation metrics
corr_matrix = classify_final_df.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

In [ ]:
# Dataframe describe to show averages, min, max for per_capita_10k
print(f'Dataframe summary: Stations per 10k people')
print(final_df_2025['per_capita_10k'].describe())

In [ ]:
# Show correlation metrics
corr_matrix = final_df_2025.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

In [ ]:
# Summary of final dataframe
print(f'Dataframe summary for distance to neighbor')
final_df_2025.describe()

### Historical analysis of Population

Review previous year and perform analysis comparing 2025 vs previous years.

In [ ]:
from helpers.grids import grid_assign_stations

# Download 2016 population data - 1km resolution
# https://hub.worldpop.org/geodata/summary?id=77814

# Define array out
grid_array_2016 = []

# Create 2016 grid_array and assign 2016 stations
stations_2016 = list_stations(2016)
grid_array_2016 = grid_assign_stations(coastline_h3, stations_2016, grid_array_2016)

# Update centroids for 2016 neighbors
grid_array_2016 = assign_grid_centroid_neighbors(stations_2016, grid_array_2016)

# Call function for current 2016 raster data
raster_2016 = 'public-datasets/jpn_pop_2016_CN_1km_R2025A_UA_v1.tif'
total_pop_2016 = population_assign_grid_array(grid_array_2016, raster_2016)
print(f'total population for 2016: {total_pop_2016}')

# Create final dataframe for 2016
final_df_2016 = grid_array_create_df(grid_array_2016)

# Show population change from 2016 to 2025
print(f'Population difference from 2016 > 2025: {total_pop_2025 - total_pop_2016}')

In [ ]:
import numpy as np
import rasterio

# Define raster files for population
import numpy as np
import rasterio

# Define raster files for population
raster_2016 = 'public-datasets/jpn_pop_2016_CN_1km_R2025A_UA_v1.tif'
raster_2025 = 'public-datasets/jpn_pop_2025_CN_1km_R2025A_UA_v1.tif'

# Read in files
with rasterio.open(raster_2016) as src1:
    data_2016 = src1.read(1, masked=True)
    meta = src1.meta.copy()
    nodata_2016 = src1.nodata

with rasterio.open(raster_2025) as src2:
    data_2025 = src2.read(1, masked=True)
    nodata_2025 = src2.nodata

# Update meta for output
meta.update(dtype=rasterio.float32, count=1, nodata=-9999.0)

# Calculate difference (2025 - 2016 = change)
data_difference = (data_2025 - data_2016).astype(np.float32)
data_difference_filled = data_difference.filled(-9999.0)

# Save new tif file
with rasterio.open('data/population-change.tif', 'w', **meta) as dst:
    dst.write(data_difference_filled, 1)

In [ ]:
# Grid object list
grid_array_difference = []

# Read in population difference tif (2016 - 2025)
raster_diff = 'data/population-change.tif'

# Assign stations and return the grid_array for 2016 - 2025
grid_array_difference = grid_assign_stations(coastline_h3, stations_list_2025, grid_array_difference)

# Assign population difference grid by grid
total_pop_difference = population_assign_grid_array(grid_array_difference, raster_diff)

# Create final dataframe for analysis
final_df_difference = grid_array_create_df(grid_array_difference)

# Define data array for heatmap
increase_pop = []
decrease_pop = []

# Iterate over station list and map on folium
for index, row in final_df_difference.iterrows():
    centroid = row['centroid']
    picked = row['population']

    # parse out rows
    latitude, longitude = centroid

    # Parse out NaN and only positive values
    if pd.notna(picked) and picked > 0:

        # Create data
        increase_pop.append([latitude, longitude, abs(picked)])

    elif pd.notna(picked) and picked < 0:

        # Create data - invert picked to show heatmap for negative values
        decrease_pop.append([latitude, longitude, abs(picked)])

In [ ]:
# Show population increase

# Generate folium map for coastline clipping and box - centered on Toyama
m = folium.Map(location=[36.70,137.21], tiles="Cartodb dark_matter", zoom_start=7)

# Add to map heatmap
HeatMap(increase_pop, min_opacity=0.1, radius=15, blur=20).add_to(m)

# Map the stations list
m

In [ ]:
# Show population decrease

# Generate folium map for coastline clipping and box - centered on Toyama
m2 = folium.Map(location=[36.70,137.21], tiles="Cartodb dark_matter", zoom_start=7)

# Add to map heatmap
HeatMap(decrease_pop, min_opacity=0.1, radius=15, blur=20).add_to(m2)

# Map the stations list
m2

In [ ]:
# Dataframe summary for population loss
final_df_difference.describe()

In [ ]:
# Show correlation metrics
corr_matrix = final_df_difference.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

### Merge the two dataframes

Merge 2016 and 2025 final dataframes

In [ ]:
# Merge the two final dataframes and add showing which lines is from what year

# Add column to each df
final_df_2016['year'] = 2016
final_df_2025['year'] = 2025

# Merge the two population dataframes
final_df_combined = pd.concat([final_df_2016, final_df_2025], ignore_index=True)

In [ ]:
# Show correlation metrics
corr_matrix = final_df_combined.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

In [ ]:
# Create population dataframes

population_2016_df = final_df_combined[final_df_combined['year'] == 2016]['population']
population_2025_df = final_df_combined[final_df_combined['year'] == 2025]['population']

### Identify underserved grids

Show grids with low train coverage

### Historical analysis of Train stations

Review previous year and perform analysis comparing 2025 vs a previous year 2016.

In [ ]:
# Show train station count difference from 2016 to 2025 count
station_count_2016 = final_df_2016['station_count'].sum()
station_count_2025 = final_df_2025['station_count'].sum()
print(f'Station change from 2016 to 2025: {station_count_2025 - station_count_2016}')

# Bar chart showing changes from 2016 to 2025
pyplot.xlabel('Year')
pyplot.ylabel("Station count")
pyplot.ylim(7000, 8500)
pyplot.bar(['2016', '2025'], [station_count_2016, station_count_2025])
pyplot.show()

In [ ]:
# Define folium map - centered on Toyama
m = folium.Map(location=[36.70,137.21], zoom_start=6)

# Filter 2016 and 2025

# Aggregate counts per h3_id per year
agg_df = final_df_combined.groupby(['h3_id', 'year'], as_index=False)['station_count'].sum()

# Pivot the aggregated data
pivot_df = agg_df.pivot(index='h3_id', columns='year', values='station_count').reset_index()
pivot_df = pivot_df.rename(columns={2016: 'station_count_2016', 2025: 'station_count_2025'})

# Fill missing values with 0
pivot_df['station_count_2016'] = pivot_df['station_count_2016'].fillna(0)
pivot_df['station_count_2025'] = pivot_df['station_count_2025'].fillna(0)

# Compute the difference
pivot_df['station_count_diff'] = pivot_df['station_count_2025'] - pivot_df['station_count_2016']

# Merge difference back into the main dataframe
final_df_combined = final_df_combined.merge(
    pivot_df[['h3_id', 'station_count_diff']],
    on='h3_id',
    how='left'
)

# Iterate over station list and map on folium
for index, row in final_df_combined.iterrows():
    latitude, longitude = row['centroid']
    if row['station_count_diff'] != 0:
        folium.Marker(
            location=[latitude, longitude],
            popup=row['centroid'],
            tooltip=row['station_count_diff']
        ).add_to(m)

# Map the stations list
m

In [ ]:
# Okinawa control grid - 1421

# https://en.wikipedia.org/wiki/Category:Railway_stations_in_Japan_closed_in_2020

# Define folium map - centered on Okinawa - 26°20′3″N 127°48′20″E
m = folium.Map(location=[26.203, 127.4820], zoom_start=10)

# Iterate over station list and map on folium
for index, row in final_df_combined.iterrows():
    latitude, longitude = row['centroid']
    if row['station_count_diff'] != 0:
        folium.Marker(
            location=[latitude, longitude],
            popup=row['station_count_diff'],
            tooltip=row['station_count_diff']
        ).add_to(m)

# Map the stations list
m

In [ ]:
# Extension (Opened Oct 1, 2019): Ishimine, Kyozuka, Urasoe-Maeda, Tedako-Uranishi.
result = final_df_combined.query("h3_id == 1421")['station_count_diff'].iloc[0]

# Assert for Okinawa changes. should be +4
assert result == 4
print(f'Train station change in Okinawa: {result}')

In [ ]:
# Graph station count for the entire dataframe difference (2016 - 2025)
final_df_combined.query('year == 2025')['station_count_diff'].plot(
    ylabel="station count change",
    xlabel="grid_id"
)

In [ ]:
# Explain the above dip

# Hirosihama =- id 737  (34.39353771718395, 132.36377649140655)

# https://en.wikipedia.org/wiki/Sank%C5%8D_Line

# Due to declining population, this closed the line roughly 30 stations

# Did not see a relative spike in population loss over those years

In [ ]:
# Pivot population by h3_id and year

# Mask for 2025 rows
mask_2025 = final_df_combined['year'] == 2025
pop_diff_map = final_df_difference.set_index('h3_id')['population']

# Update only 2025 rows
final_df_combined.loc[mask_2025, 'population_count_diff'] = final_df_combined.loc[mask_2025, 'h3_id'].map(pop_diff_map)

# Fill any missing values just in case
final_df_combined['population_count_diff'] = final_df_combined['population_count_diff'].fillna(0)

# Graph population for the entire dataframe difference (2016 - 2025)
final_df_combined.query('year == 2025')['population_count_diff'].plot(
    ylabel="population count change",
    xlabel="grid_id",
)

In [ ]:
# Map all h3 ids and population change and station hange

pop_change = final_df_combined['population_count_diff']
station_change = final_df_combined['station_count_diff']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Drop na values
df_clean = final_df_combined.dropna(subset=['population_count_diff', 'station_count_diff'])

# Plot
sns.lmplot(
    x='population_count_diff',
    y='station_count_diff',
    data=df_clean,
    height=6,
    aspect=1.2
)

plt.xlabel('Population Change')
plt.ylabel('Station Count Change')
plt.title('Relationship with Trend Line')
plt.show()

In [ ]:
df_no_stations = final_df_2025.query('station_count == 0')['centroid_to_station_km']

# For those grids with no stations,, list the distance
df_no_stations.plot(
    xlabel="grid id",
    ylabel= "distance in km",
    title="Grids with no stations and distance to nearest station"
)

### Historical analysis of Train stations Impact on population

Review previous year and perform analysis comparing 2025 vs a previous year.

In [ ]:
import seaborn as sns

# Show correlation metrics
corr_matrix = final_df_combined.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, vmax=1, annot=True, linewidths=.5)
plt.xticks(rotation=30, horizontalalignment='right')
plt.show()

### Important Maps



In [ ]:
# Centroids within 5km of a station

df_temp = final_df_2025.query('centroid_to_station_km < 5')

# Histogram for distance to nearest station from a grid centroid
pyplot.figure(figsize=(7, 3))
pyplot.hist(df_temp["centroid_to_station_km"].dropna(), bins=50)
pyplot.title("Distance to nearest station (km)")
pyplot.xlabel("km")
pyplot.ylabel("count")
pyplot.tight_layout()
pyplot.show()

In [ ]:
from matplotlib.ticker import LogLocator, FuncFormatter

# Centroids within 10km of a station
centroid_df = final_df_2025.query('centroid_to_station_km > 10')

# Histogram for distance to nearest station from a grid centroid
fig, ax = plt.subplots(figsize=(7, 3))
pyplot.hist(centroid_df["centroid_to_station_km"].dropna(), bins=50)
pyplot.title("Distance to nearest station (km)")
pyplot.xlabel("km")
pyplot.yscale('log')

# Rescale axis so it's logarithmic scale but in base 10
ax = pyplot.gca()
ax.yaxis.set_major_locator(LogLocator(base=10))

# Plain-number tick labels (10, 100, etc.)
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda y, _: f"{int(y):,}" if y >= 1 and float(y).is_integer() else f"{y:g}")
)

# Draw table
ax.set_title("Distance to nearest station (km)")
ax.set_xlabel("km")
ax.set_ylabel("count (log scale)")
fig.tight_layout()
plt.show()

In [ ]:
# Centroids within 5km of a station

df_temp = final_df_2025.query('centroid_to_station_km < 5')

# Histogram for distance to nearest station from a grid centroid
pyplot.figure(figsize=(7, 3))
pyplot.hist(df_temp["per_capita_10k"].dropna(), bins=50)
pyplot.title("Distance to nearest station (km)")
pyplot.xlabel("km")
pyplot.ylabel("count")
pyplot.tight_layout()
pyplot.show()

In [ ]:
# Query where station count is 0 graph distance to nearest

df_temp = final_df_2025.query('station_count == 0')

# Histogram for distance to nearest station from a grid centroid
pyplot.figure(figsize=(7, 3))
pyplot.hist(df_temp["centroid_to_station_km"].dropna(), bins=50)
pyplot.title("Distance to nearest station (km) where no station exists")
pyplot.xlabel("km")
pyplot.ylabel("count")
pyplot.tight_layout()
pyplot.show()

In [ ]:
# Map the grids where the nearest station is 100km away, primarily finds islands

centroid_to_station_km_outliers = final_df_2025.query('centroid_to_station_km > 100')

In [ ]:
# Map the grids where the nearest station is 100km away, primarily finds islands

# Define folium map - centered on Toyama
m = folium.Map(location=[36.70,137.21], zoom_start=5)

# Iterate over station list and map on folium
for _, row in centroid_to_station_km_outliers.iterrows():
    lat, lon = row['centroid']
    folium.Marker(
		location=[lat, lon],
		popup=f"{row['h3_id']} | {row['centroid_to_station_km']:.1f} km",
		tooltip=f"pop={row['population']:,} | d={row['centroid_to_station_km']:.1f} km",
	).add_to(m)

# add grid
folium.GeoJson(coastline_h3).add_to(m)

# Map the stations list
m